# 12 — Repair Mechanism (Graded Focus)

**Maps to:** `report/Chapters/Task2.tex` §`T2:RepairDesign` **and** `report/Chapters/Task3.tex` §`T3:RepairCritique`.  
**Ticket:** TICKET-12.

Step-by-step:
1. scan, mark first occurrence of each city;
2. flag duplicates;
3. identify missing cities;
4. replace duplicates with missing — three insertion variants: random / position-based / nearest-neighbour.

Show repair on:
- no-duplicate input (identity);
- single duplicate;
- multiple duplicates;
- edge cases (all-same, reversed).

## Setup

The helper functions below follow the same conventions as the earlier notebooks: `load_tsp` parses a TSPLIB file into coordinates and a Euclidean distance matrix (notebook 04), `tour_length` computes the closed-tour distance (notebook 05), and `is_valid_permutation` checks tour feasibility (notebook 05).

In [1]:
from pathlib import Path

import numpy as np
import tsplib95


def load_tsp(path):
    problem     = tsplib95.load(str(path))
    nodes       = list(problem.get_nodes())
    coords      = np.array(
        [problem.node_coords[node] for node in nodes],
        dtype=np.float64
    )
    diff        = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))
    return coords, dist_matrix


def tour_length(chromosome, dist_matrix):
    chromosome = np.asarray(chromosome)
    return dist_matrix[chromosome, np.roll(chromosome, -1)].sum()


def is_valid_permutation(chromosome, n_cities):
    return (
        len(chromosome) == n_cities
        and set(chromosome) == set(range(n_cities))
    )

## Repair Design

Operators such as naive one-point crossover copy segments from two parents without coordinating their contents. The offspring therefore frequently contains **duplicate** cities (visited twice) and **missing** cities (never visited), which violates the permutation constraint of the TSP. The repair mechanism transforms such an infeasible chromosome into a feasible tour in four steps:

1. **Scan** the chromosome left to right and mark the *first occurrence* of each city. First occurrences are always kept, so as much of the parent's ordering as possible is preserved.
2. **Flag duplicates** — every position whose city has already been seen earlier in the scan is recorded as a duplicate slot.
3. **Identify missing cities** — the set difference between all cities $\{0, \dots, n-1\}$ and the cities present in the chromosome.
4. **Replace** each duplicate slot with a missing city. The pairing between duplicate slots and missing cities is decided by one of three selectable insertion variants:
   - **Random insertion** (`strategy="random"`) — missing cities are shuffled and assigned to the duplicate slots in arbitrary order. Unbiased and cheap, but ignores tour quality entirely.
   - **Position-based insertion** (`strategy="position"`) — missing cities in ascending index order are assigned to duplicate slots in ascending position order. Fully deterministic: the same infeasible chromosome always repairs to the same tour, which is useful for reproducibility and debugging.
   - **Nearest-neighbour insertion** (`strategy="nearest"`) — duplicate slots are processed left to right, and each slot receives the *closest* still-unassigned missing city with respect to the city immediately preceding the slot in the (partially repaired) tour. This is a greedy, distance-aware variant that requires the distance matrix.

The number of duplicate slots always equals the number of missing cities, so every variant produces a complete assignment. By construction, the repaired chromosome contains every city exactly once and is therefore always a valid permutation.

**Complexity.** Steps 1–3 are a single $O(n)$ scan. Random and position-based replacement add $O(m)$ for $m$ duplicates, so the whole repair is $O(n)$. Nearest-neighbour replacement compares every remaining missing city for each duplicate slot, adding $O(m^2)$ — still negligible against the $O(n^2)$ fitness evaluations of the GA itself.

In [2]:
def diagnose(chromosome, n_cities):
    """Steps 1-3: scan for first occurrences, flag duplicate positions, list missing cities."""
    seen          = set()
    dup_positions = []
    for pos, city in enumerate(chromosome):
        if city in seen:
            dup_positions.append(pos)
        else:
            seen.add(int(city))
    missing = sorted(set(range(n_cities)) - seen)
    return dup_positions, missing


def repair(chromosome, n_cities, dist_matrix=None, strategy="random", rng=None):
    """Return a valid permutation derived from a possibly infeasible chromosome.

    strategy: "random" | "position" | "nearest"
    """
    tour = np.asarray(chromosome).copy()
    dup_positions, missing = diagnose(tour, n_cities)

    if not dup_positions:
        return tour

    if strategy == "random":
        if rng is None:
            rng = np.random.default_rng()
        replacements = list(rng.permutation(missing))
        for pos, city in zip(dup_positions, replacements):
            tour[pos] = city

    elif strategy == "position":
        # dup_positions and missing are both ascending: deterministic pairing
        for pos, city in zip(dup_positions, missing):
            tour[pos] = city

    elif strategy == "nearest":
        if dist_matrix is None:
            raise ValueError("strategy='nearest' requires dist_matrix")
        remaining = list(missing)
        for pos in dup_positions:
            prev_city = tour[pos - 1]  # pos=0 wraps to the last city: closed tour
            nearest   = min(remaining, key=lambda city: dist_matrix[prev_city, city])
            tour[pos] = nearest
            remaining.remove(nearest)

    else:
        raise ValueError(f"unknown strategy: {strategy!r}")

    assert is_valid_permutation(tour, n_cities), "repair produced an invalid permutation"
    return tour

## Worked Example (step-by-step trace)

An 8-city chromosome `[3, 1, 4, 1, 5, 2, 6, 3]` is repaired below with each step printed explicitly. City `1` appears at positions 1 and 3, and city `3` appears at positions 0 and 7, so positions 3 and 7 are duplicate slots; cities `0` and `7` never appear and are missing.

In [3]:
example   = np.array([3, 1, 4, 1, 5, 2, 6, 3])
n_example = 8

print(f"Infeasible chromosome : {example}")

# Steps 1-2: scan and flag duplicates
seen = set()
for pos, city in enumerate(example):
    status = "DUPLICATE" if city in seen else "first occurrence"
    seen.add(int(city))
    print(f"  pos {pos}: city {city} -> {status}")

dup_positions, missing = diagnose(example, n_example)
print(f"\nStep 2 - duplicate slots  : positions {dup_positions}")
print(f"Step 3 - missing cities   : {missing}")

# Step 4: replacement under each insertion variant
octagon_coords = np.array(
    [[0, 0], [2, 0], [4, 0], [4, 2], [4, 4], [2, 4], [0, 4], [0, 2]],
    dtype=np.float64
)
diff         = octagon_coords[:, np.newaxis, :] - octagon_coords[np.newaxis, :, :]
example_dist = np.sqrt((diff ** 2).sum(axis=-1))

rng = np.random.default_rng(42)
for strategy in ("random", "position", "nearest"):
    repaired = repair(example, n_example, dist_matrix=example_dist, strategy=strategy, rng=rng)
    print(f"\nStep 4 ({strategy:>8}) : {repaired}  valid={is_valid_permutation(repaired, n_example)}")

Infeasible chromosome : [3 1 4 1 5 2 6 3]
  pos 0: city 3 -> first occurrence
  pos 1: city 1 -> first occurrence
  pos 2: city 4 -> first occurrence
  pos 3: city 1 -> DUPLICATE
  pos 4: city 5 -> first occurrence
  pos 5: city 2 -> first occurrence
  pos 6: city 6 -> first occurrence
  pos 7: city 3 -> DUPLICATE

Step 2 - duplicate slots  : positions [3, 7]
Step 3 - missing cities   : [0, 7]

Step 4 (  random) : [3 1 4 7 5 2 6 0]  valid=True

Step 4 (position) : [3 1 4 0 5 2 6 7]  valid=True

Step 4 ( nearest) : [3 1 4 7 5 2 6 0]  valid=True


All three variants keep the first occurrences (`3, 1, 4, 5, 2, 6`) in their original positions and differ only in how the missing cities `0` and `7` are placed into the duplicate slots at positions 3 and 7:

- **random** shuffles the missing cities before assignment, so the pairing depends on the random seed;
- **position** assigns missing city `0` to the first duplicate slot and `7` to the second, deterministically;
- **nearest** looks at the city preceding each slot — position 3 follows city `4` and position 7 follows city `6` — and picks whichever unassigned missing city is closer on the example layout.

## Unit Tests

The required cases from the ticket: no duplicates (repair must be the identity), a single duplicate, multiple duplicates, and edge cases (all-same chromosome, reversed-but-valid tour). A property test then corrupts 500 random tours and checks that every strategy always returns a valid permutation — the core acceptance criterion.

In [4]:
rng = np.random.default_rng(42)

# Case 1: no duplicates -> identity
valid_tour = np.array([4, 2, 0, 3, 1])
for strategy in ("random", "position", "nearest"):
    out = repair(valid_tour, 5, dist_matrix=example_dist[:5, :5], strategy=strategy, rng=rng)
    assert np.array_equal(out, valid_tour), f"{strategy}: valid tour must pass through unchanged"
print("Case 1 passed: valid tour is returned unchanged by all strategies.")

# Case 2: single duplicate
single_dup = np.array([0, 1, 2, 1, 4])          # city 1 duplicated, city 3 missing
out = repair(single_dup, 5, strategy="position")
assert is_valid_permutation(out, 5)
assert out[3] == 3, f"expected missing city 3 at position 3, got {out}"
print("Case 2 passed: single duplicate replaced by the missing city.")

# Case 3: multiple duplicates
multi_dup = np.array([3, 1, 4, 1, 5, 2, 6, 3])  # duplicates at positions 3 and 7
for strategy in ("random", "position", "nearest"):
    out = repair(multi_dup, 8, dist_matrix=example_dist, strategy=strategy, rng=rng)
    assert is_valid_permutation(out, 8), f"{strategy} failed on multiple duplicates"
print("Case 3 passed: multiple duplicates repaired by all strategies.")

# Case 4 (edge): all-same chromosome -- maximal infeasibility
all_same = np.zeros(8, dtype=int)
for strategy in ("random", "position", "nearest"):
    out = repair(all_same, 8, dist_matrix=example_dist, strategy=strategy, rng=rng)
    assert is_valid_permutation(out, 8), f"{strategy} failed on all-same chromosome"
print("Case 4 passed: all-same chromosome repaired to a full permutation.")

# Case 5 (edge): reversed valid tour -- still valid, must be identity
reversed_tour = np.arange(8)[::-1].copy()
out = repair(reversed_tour, 8, strategy="position")
assert np.array_equal(out, reversed_tour)
print("Case 5 passed: reversed (valid) tour is returned unchanged.")

# Case 6 (edge): position-based repair is deterministic
a = repair(multi_dup, 8, strategy="position")
b = repair(multi_dup, 8, strategy="position")
assert np.array_equal(a, b)
print("Case 6 passed: position-based repair is deterministic.")

# Case 7 (edge): nearest strategy without dist_matrix must raise
try:
    repair(multi_dup, 8, strategy="nearest")
    raise AssertionError("expected ValueError")
except ValueError:
    print("Case 7 passed: strategy='nearest' without dist_matrix raises ValueError.")

# Property test: 500 random corruptions, every strategy always yields a valid permutation
n_cities = 20
coords_p = rng.random((n_cities, 2)) * 100
diff_p   = coords_p[:, np.newaxis, :] - coords_p[np.newaxis, :, :]
dist_p   = np.sqrt((diff_p ** 2).sum(axis=-1))

for trial in range(500):
    corrupted = rng.integers(0, n_cities, size=n_cities)   # arbitrary, usually infeasible
    for strategy in ("random", "position", "nearest"):
        out = repair(corrupted, n_cities, dist_matrix=dist_p, strategy=strategy, rng=rng)
        assert is_valid_permutation(out, n_cities), (
            f"trial {trial}, {strategy}: invalid output {out} for input {corrupted}"
        )
print("Property test passed: 500 random corruptions x 3 strategies -> always a valid permutation.")

Case 1 passed: valid tour is returned unchanged by all strategies.
Case 2 passed: single duplicate replaced by the missing city.
Case 3 passed: multiple duplicates repaired by all strategies.
Case 4 passed: all-same chromosome repaired to a full permutation.
Case 5 passed: reversed (valid) tour is returned unchanged.
Case 6 passed: position-based repair is deterministic.
Case 7 passed: strategy='nearest' without dist_matrix raises ValueError.
Property test passed: 500 random corruptions x 3 strategies -> always a valid permutation.


## Demo on kroA100

To exercise the repair mechanism under realistic conditions, infeasible offspring are produced by a **naive one-point crossover** — the operator analysed in notebook 08 that motivates repair in the first place. The child copies the first parent up to a cut point and the second parent after it, which almost always duplicates some cities and drops others on a 100-city instance.

Each strategy then repairs 30 such offspring (fixed seed), and the resulting tour lengths are compared.

In [5]:
def naive_one_point_crossover(parent_a, parent_b, cut):
    return np.concatenate([parent_a[:cut], parent_b[cut:]])


coords, dist_matrix = load_tsp(Path('../data/TSP-dataset/kroA100.tsp'))
n = len(coords)

rng = np.random.default_rng(42)

# one offspring, inspected in detail
parent_a = rng.permutation(n)
parent_b = rng.permutation(n)
child    = naive_one_point_crossover(parent_a, parent_b, cut=n // 2)

dup_positions, missing = diagnose(child, n)
print(f"Naive one-point crossover offspring (cut at {n // 2}):")
print(f"  valid permutation : {is_valid_permutation(child, n)}")
print(f"  duplicate slots   : {len(dup_positions)}")
print(f"  missing cities    : {len(missing)}")

Naive one-point crossover offspring (cut at 50):
  valid permutation : False
  duplicate slots   : 24
  missing cities    : 24


In [6]:
n_offspring = 30
rng         = np.random.default_rng(42)
lengths     = {"random": [], "position": [], "nearest": []}
infeasible_count = 0

for _ in range(n_offspring):
    pa    = rng.permutation(n)
    pb    = rng.permutation(n)
    cut   = int(rng.integers(1, n - 1))
    child = naive_one_point_crossover(pa, pb, cut)
    if not is_valid_permutation(child, n):
        infeasible_count += 1
    for strategy in lengths:
        repaired = repair(child, n, dist_matrix=dist_matrix, strategy=strategy, rng=rng)
        assert is_valid_permutation(repaired, n)
        lengths[strategy].append(tour_length(repaired, dist_matrix))

print(f"Offspring infeasible before repair : {infeasible_count}/{n_offspring}")
print(f"All repaired tours valid           : True")
print(f"\n{'Strategy':<12} {'mean length':>12} {'min':>12} {'max':>12}")
for strategy, vals in lengths.items():
    vals = np.array(vals)
    print(f"{strategy:<12} {vals.mean():>12.0f} {vals.min():>12.0f} {vals.max():>12.0f}")
print(f"\nKnown optimal tour length: 21,282")

Offspring infeasible before repair : 30/30
All repaired tours valid           : True

Strategy      mean length          min          max
random             170436       153630       190396
position           171245       155086       188894
nearest            151149       130498       171004

Known optimal tour length: 21,282


## Discussion

Every naive-crossover offspring was infeasible before repair, confirming the analysis from notebook 08, and every repaired tour passed the validity check across all strategies — the central guarantee of the mechanism.

The comparison also previews a trade-off that matters for the GA experiments (notebooks 16–17):

- **Random** and **position-based** insertion preserve feasibility but pay no attention to distance, so repaired tours remain near random-tour quality on kroA100. They are *neutral* repairs: cheap, unbiased, and they leave optimisation pressure entirely to selection.
- **Nearest-neighbour** insertion produces noticeably shorter repaired tours by greedily connecting each repaired slot to a nearby city. This injects local optimisation into the repair step itself — which can accelerate early convergence but also adds greedy bias and may reduce population diversity, a point to examine with the diversity metrics from notebook 14.

These properties — guaranteed feasibility, variant-dependent bias, and the exploration/exploitation tension — are the basis of the critical analysis in `Task3.tex` §`T3:RepairCritique` (TICKET-21).